# Edge Case 02 — Tool Error Envelopes

**No API key required. Target: under 2 minutes.**

`DeterministicToolExecutor` is the safety boundary between the model and your tools.
Regardless of how a tool fails, the result is always a typed `ToolResult` envelope:
- `status` — one of `SUCCESS`, `ERROR`, `BLOCKED`, `TIMEOUT`, `CANCELLED`
- `error` — a `NormalizedError` with `code`, `category`, `message`, `retryable`
- `audit` — `ToolAudit` with `correlation_id` for traceability
- `result` — populated on `SUCCESS`, `None` on failure

The raw exception stack trace is **never** exposed to the model.

In [ ]:
import pathlib
import sys, os

_repo = pathlib.Path(os.path.abspath(".."))
sys.path.insert(0, str(_repo))
_contracts_src = _repo / "packages" / "eXo_adapters" / "packages" / "exo-brain-core-contracts" / "src"
if _contracts_src.is_dir():
    sys.path.insert(0, str(_contracts_src))

try:
    from dotenv import load_dotenv
    load_dotenv("../.env", override=False)
except ImportError:
    pass

from src.tools.executor import DeterministicToolExecutor
from src.tools.registry import ToolRegistry, ToolDescriptor
from src.schemas.tool_io import (
    ToolCallContext, ToolResult, ToolStatus, NormalizedError, RiskTier,
)
from src.policies.middleware import DeterministicFirstPolicyMiddleware

registry = ToolRegistry()
policy = DeterministicFirstPolicyMiddleware()
executor = DeterministicToolExecutor(registry=registry, policy=policy)

def make_call(tool_name: str, arguments: dict | None = None) -> ToolCallContext:
    return ToolCallContext(
        schema_version="1.0",
        call_id=f"call-{tool_name}-001",
        session_id="session-edge-02",
        run_id="run-edge-02",
        job_id="job-edge-02",
        task_id="task-edge-02",
        agent_id="agent-edge-02",
        provider_id="demo",
        tool_name=tool_name,
        arguments=arguments or {},
        tenant_id="tenant-edge",
        risk_tier=RiskTier.LOW,
        is_state_changing=False,
    )

print("executor :", type(executor).__name__)
print("registry :", type(registry).__name__)

## Part 1 — Register tools with different failure modes

Three tools:
1. `raises_value_error` — raises `ValueError` on every call
2. `raises_runtime_error` — raises `RuntimeError` on every call
3. `success_tool` — returns a valid `dict`

All are registered with `RiskTier.LOW` so policy does not block them.

In [ ]:
def _raises_value_error(**kwargs):
    raise ValueError("Bad argument: 'x' must be positive")

def _raises_runtime_error(**kwargs):
    raise RuntimeError("Service unavailable: upstream timeout")

def _success_tool(x: int = 0) -> dict:
    return {"doubled": x * 2, "operation": "double"}

registry.register(ToolDescriptor(
    name="raises_value_error",
    handler=_raises_value_error,
    risk_tier=RiskTier.LOW,
    is_state_changing=False,
    description="Always raises ValueError.",
))
registry.register(ToolDescriptor(
    name="raises_runtime_error",
    handler=_raises_runtime_error,
    risk_tier=RiskTier.LOW,
    is_state_changing=False,
    description="Always raises RuntimeError.",
))
registry.register(ToolDescriptor(
    name="success_tool",
    handler=_success_tool,
    risk_tier=RiskTier.LOW,
    is_state_changing=False,
    description="Returns doubled value.",
))

print("Registered tools:", registry.list_tools())

## Part 2 — Execute each tool and show the error envelope

Every failure is wrapped in a `ToolResult(status=ERROR)` with a structured `NormalizedError`.

In [ ]:
def show_result(label: str, result: ToolResult) -> None:
    icon = "✅" if result.status == ToolStatus.SUCCESS else "❌"
    print(f"{icon} {label}")
    print(f"   status   : {result.status.value}")
    print(f"   result   : {result.result}")
    if result.error:
        print(f"   error.code     : {result.error.code}")
        print(f"   error.category : {result.error.category}")
        print(f"   error.message  : {result.error.message[:80] if result.error.message else None}")
        print(f"   error.retryable: {result.error.retryable}")
    if result.audit:
        print(f"   audit.correlation_id: {result.audit.correlation_id[:20]}...")
    print()

r_ve = executor.execute(make_call("raises_value_error"))
r_re = executor.execute(make_call("raises_runtime_error"))

show_result("raises_value_error", r_ve)
show_result("raises_runtime_error", r_re)

# Both must be ERROR, never SUCCESS
assert r_ve.status == ToolStatus.ERROR, f"Expected ERROR, got {r_ve.status}"
assert r_re.status == ToolStatus.ERROR, f"Expected ERROR, got {r_re.status}"
assert r_ve.result is None, "result must be None on error"
assert r_re.result is None, "result must be None on error"
print("PASS — both failures wrapped as ToolResult(status=ERROR)")

## Part 3 — Error envelope fields in detail

`NormalizedError` gives the model a structured, safe error representation.
The raw exception type and stack trace are never included in `result.result`.

In [ ]:
# Verify NormalizedError fields are structured
err = r_ve.error
assert err is not None
assert isinstance(err.code, str) and len(err.code) > 0
assert isinstance(err.category, str)
assert isinstance(err.message, str)
assert isinstance(err.retryable, bool)

print("NormalizedError fields for raises_value_error:")
print(f"  code     : {err.code}")
print(f"  category : {err.category}")
print(f"  message  : {err.message[:100]}")
print(f"  retryable: {err.retryable}")

# The raw stack trace is NOT in result.result
assert r_ve.result is None, "Stack trace must never appear in result.result"
print()
print("PASS — NormalizedError is structured; stack trace never in result.result")

## Part 4 — Tool not found

Calling a tool that was never registered produces `TOOL_NOT_FOUND` error code.

In [ ]:
r_missing = executor.execute(make_call("nonexistent_tool"))
show_result("nonexistent_tool", r_missing)

assert r_missing.status == ToolStatus.ERROR
assert r_missing.error is not None
assert r_missing.error.code == "TOOL_NOT_FOUND"
print("PASS — TOOL_NOT_FOUND error code returned for unregistered tool")

## Part 5 — Success case for comparison

A tool that returns a valid `dict` produces `ToolResult(status=SUCCESS)` with:
- `result` populated with the tool's return value
- `audit.correlation_id` set for traceability

In [ ]:
r_ok = executor.execute(make_call("success_tool", {"x": 21}))
show_result("success_tool (x=21)", r_ok)

assert r_ok.status == ToolStatus.SUCCESS, f"Expected SUCCESS, got {r_ok.status}"
assert r_ok.result is not None
# The executor wraps the handler's return value under the "value" key
tool_output = r_ok.result.get("value", r_ok.result)
doubled = tool_output.get("doubled") if isinstance(tool_output, dict) else r_ok.result.get("doubled")
assert doubled == 42, f"Expected doubled=42, got result={r_ok.result}"
assert r_ok.audit is not None
assert r_ok.audit.correlation_id  # non-empty

print("PASS — success tool returns ToolResult(status=SUCCESS, doubled=42)")
print(f"       result (raw)       : {r_ok.result}")
print(f"       audit.correlation_id: {r_ok.audit.correlation_id[:20]}...")

## Part 6 — Validation error: missing required context

A `ToolCallContext` with `schema_version != "1.0"` fails validation before execution.

In [ ]:
invalid_call = ToolCallContext(
    schema_version="0.9",   # wrong version
    call_id="call-invalid",
    session_id="session-edge-02",
    run_id="run-edge-02",
    job_id="job-edge-02",
    task_id="task-edge-02",
    agent_id="agent-edge-02",
    provider_id="demo",
    tool_name="success_tool",
    arguments={"x": 5},
    tenant_id="tenant-edge",
)

r_invalid = executor.execute(invalid_call)
show_result("invalid schema_version", r_invalid)

assert r_invalid.status == ToolStatus.ERROR
assert r_invalid.error.code == "TOOL_CALL_VALIDATION_ERROR"
print("PASS — schema validation error wrapped as ToolResult(status=ERROR)")

print()
print("All edge_02 scenarios: PASS")
print("DeterministicToolExecutor is a safety boundary — every outcome is a typed ToolResult.")